**Convnet training functions**
- Folder name can be set according to your need.
- Remember to **update the training settings** for different training configurations.
    - **small model: {nFeat, ReduceRatio, nResBlock, nEpochs}={32, 1, 3, 15}**
    - **large model: {nFeat, ReduceRatio, nResBlock, nEpochs}={64, 2, 8, 120}**
- It is recommended to use the CPU for debugging before running the full training process.
- Training with a GPU may be interrupted midway. In that case, you can use the last code block to restore the checkpoint and resume training.
    - If your GPU usage reaches the limit, you can also switch to the CPU and complete the remaining training process.
- To avoid reaching the usage limit, it is recommended to disconnect the runtime when you are not using it.
    - 執行階段 → 中斷連線並刪除執行階段

In [ ]:
from pathlib import Path
import os


def find_hw03_root():
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / 'optimization-based').exists() and (path / 'convnet-based').exists():
            return Path(os.path.relpath(path, cwd))

        hw03 = path / 'hw03'
        if (hw03 / 'optimization-based').exists() and (hw03 / 'convnet-based').exists():
            return Path(os.path.relpath(hw03, cwd))

    raise FileNotFoundError('Run this notebook from the repository root, hw03, or an hw03 subfolder.')


FOLDER_NAME = str(find_hw03_root())


In [ ]:
%run '{FOLDER_NAME}/convnet-based/model.ipynb'
%run '{FOLDER_NAME}/convnet-based/dataset.ipynb'

===== Environment settings =====
- If you want to train the model with a GPU, please go to
    - 執行階段 → 變更執行階段類型 → 硬體加速器選 "T4 GPU" → 儲存
- If the GPU is enabled successfully, executing the following code block will display:
    - cuda available: True

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.autograd import Variable
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
from math import log10
from types import SimpleNamespace
import os
import numpy
import random

===== Training settings =====
- **Remember to modify your training setting here.**

In [ ]:
#===== Training settings =====#
args = SimpleNamespace(
    patchSize=64,           # HR image cropping (patch) size for training
    batchSize=16,           # training batch size
    epochSize=150,          # number of batches as one epoch (for validating once)

    #============ you need to modify in implementation part  ==============
    nEpochs=120,             # number of epochs for training
    nFeat=64,               # channel number of feature maps
    ReduceRatio=2,          # expansion ratio of residual block
    nResBlock=8,            # number of residual blocks

    # nEpochs=15,             # number of epochs for training
    # nFeat=32,               # channel number of feature maps
    # ReduceRatio=1,          # expansion ratio of residual block
    # nResBlock=3,            # number of residual blocks
    #===============================================================================

    nTrain=2,               # number of training images
    nVal=1,                 # number of validation images
    cuda=torch.cuda.is_available(), # use cuda?
    lr=1e-4,                # learning rate
    threads=2,              # number of threads for data loader to use, if Your OS is window, please set to 0
    seed=715,               # random seed to use
    printEvery=30,          # number of batches to print average loss
    data_dir = os.path.join(FOLDER_NAME, 'convnet-based')
)

print(args)

if args.cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

torch.manual_seed(args.seed)
if args.cuda:
    torch.cuda.manual_seed(args.seed)
random.seed(args.seed)
numpy.random.seed(args.seed)
torch.backends.cudnn.benchmark = False

===== Datasets =====
- No modification is needed for the following code.

In [ ]:
#===== Datasets =====#
def seed_worker(worker_id):
    worker_seed = args.seed
    numpy.random.seed(args.seed)
    random.seed(args.seed)

print('===> Loading datasets')
train_set = datasetTrain_speedup(args)  # call datasetTrain.__init__(train_set, args)
# train_set = datasetTrain(args)      # call datasetTrain.__init__(train_set, args)

train_data_loader = DataLoader(dataset=train_set, num_workers=args.threads, batch_size=args.batchSize, shuffle=True, worker_init_fn=seed_worker)
val_set = datasetVal(args)
val_data_loader = DataLoader(dataset=val_set, num_workers=args.threads, batch_size=1, shuffle=False, worker_init_fn=seed_worker)

===== ZebraSRNet model =====

In [ ]:
#===== ZebraSRNet model =====#
print('===> Building model')
net = ZebraSRNet(nFeat=args.nFeat, ReduceRatio=args.ReduceRatio, nResBlock=args.nResBlock)

if args.cuda:
    net = net.cuda()

===== Loss function and optimizer =====

In [ ]:
#===== Loss function and optimizer =====#
criterion = torch.nn.L1Loss()

if args.cuda:
    criterion = criterion.cuda()

optimizer = torch.optim.Adam(net.parameters(), lr=args.lr)

===== Training and validation procedures =====

In [ ]:
#===== Training and validation procedures =====#
def train(f, epoch):
    net.train()
    epoch_loss = 0
    for iteration, batch in enumerate(train_data_loader):
        # call train_set.__getitem__(idx) recursively
        varIn, varTar = Variable(batch[0]), Variable(batch[1])
        if args.cuda:
            varIn = varIn.cuda()
            varTar = varTar.cuda()

        optimizer.zero_grad()
        loss = criterion(net(varIn), varTar)
        epoch_loss += loss.data
        loss.backward()
        optimizer.step()
        if (iteration+1)%args.printEvery == 0:
            print("===> Epoch[{}]({}/{}): Avg. Loss: {:.4f}".format(epoch, iteration+1, len(train_data_loader), epoch_loss/args.printEvery))
              # len(train_data_loader) call train_set.__len__()
            f.write("===> Epoch[{}]({}/{}): Avg. Loss: {:.4f}\n".format(epoch, iteration+1, len(train_data_loader), epoch_loss/args.printEvery))
            epoch_loss = 0

In [ ]:
def validate(f):
    net.eval()
    avg_psnr = 0
    mse_criterion = torch.nn.MSELoss()
    for batch in val_data_loader:
        varIn, varTar = Variable(batch[0]), Variable(batch[1])
        if args.cuda:
            varIn = varIn.cuda()
            varTar = varTar.cuda()

        prediction = net(varIn)
        prediction[prediction>  1] =   1
        prediction[prediction<  0] =   0
        mse = mse_criterion(prediction, varTar)
        psnr = 10 * log10(1.0*1.0/mse.data)
        avg_psnr += psnr
    print("===> Avg. PSNR: {:.4f} dB".format(avg_psnr / len(val_data_loader)))
    f.write("===> Avg. PSNR: {:.4f} dB\n".format(avg_psnr / len(val_data_loader)))

===== Model saving =====

In [ ]:
#===== Model saving =====#
save_dir = f'{FOLDER_NAME}/convnet-based/model_trained'
if not os.path.isdir(save_dir):
    os.makedirs(save_dir)

def checkpoint(epoch):
    save_name = 'train_net_F{}B{}R{}_epoch_{}.pth'.format(args.nFeat, args.nResBlock, args.ReduceRatio, epoch)
    save_path = os.path.join(save_dir, save_name)
    torch.save(net, save_path)
    print("Checkpoint saved to {}".format(save_path))


===== Training start =====

In [ ]:
import time

log_path = f"{FOLDER_NAME}/convnet-based/train_net_F{args.nFeat}B{args.nResBlock}R{args.ReduceRatio}.log"

t = time.time()
with open(log_path, 'w') as f:
    f.write('training log record of F={}, B={}, R={}, random seed={}\n'.format(args.nFeat, args.nResBlock, args.ReduceRatio, args.seed))
    f.write('dataset configuration: epoch size = {}, batch size = {}, patch size = {}\n'.format(args.epochSize, args.batchSize, args.patchSize))
    print('-------')
    for epoch in range(1, args.nEpochs+1):
        train(f, epoch)
        validate(f)
        checkpoint(epoch)

print(f"Duration: {time.time() - t}")

===== Checkpoint Recovery =====
- If the training process is unexpectedly interrupted, you can use the following code to restore the checkpoint and resume training.
- Uncomment the following code block and comment out the previous one.
- Make sure that **start_epoch** and **ckpt_path** are set correctly.
- Then click "**Run All**".

In [ ]:
# #===== Checkpoint Recovery =====#

# resume_ckpt_path = f"{FOLDER_NAME}/convnet-based/model_trained/train_net_F64B8R2_epoch_10.pth"
# start_epoch = 10

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# net = torch.load(resume_ckpt_path, map_location=device, weights_only=False)
# net = net.to(device)

# optimizer = torch.optim.Adam(net.parameters(), lr=args.lr)

# log_path = f"{FOLDER_NAME}/convnet-based/train_net_F{args.nFeat}B{args.nResBlock}R{args.ReduceRatio}.log"

# with open(log_path, 'a') as f:   # use 'a' to append log file, instead of overwrite
#     f.write(f"\nResume training from epoch {start_epoch}\n")
#     for epoch in range(start_epoch, args.nEpochs + 1):
#         train(f, epoch)
#         validate(f)
#         checkpoint(epoch)
